# Exploring Meta data

This notebook explores and describes the meta data.
- Finding a paper that is nice to highlight in a presentation.
- Proportions of categorical columns.
- Distribution of numerical columns.

### Settings

In [ ]:
experiment_name = "free_1000_251013_pest_PD"

## Initialisation

In [ ]:
# meta
__author__ ="Jennefer Beenen"
__version__ = "1.0"
__email__ = "j.beenen@pl.hanze.nl"
__status__ = "Development"
# __date__ = "2025-05-06"

### Imports

In [ ]:
# imports
from pathlib import Path
import pandas as pd
from matplotlib_venn import venn3
import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np

### Load files

In [ ]:
filename = f'meta_{experiment_name}.csv'
data_folder = "../../data/"
current_year = 2025

In [ ]:
# load data
df = pd.read_csv(f'{data_folder}meta/{filename}', index_col=0)
df.info()

### Add columns `is_pdf`, `is_xml`, `cited_count_per_year`

In [ ]:
# load pmids downloaded in `data/xml_papers`
local_pdfs = {int(f.stem) for f in Path(f'{data_folder}pdf_papers/').iterdir() if f.suffixes[0] == ".pdf"}
local_xmls = {int(f.stem) for f in Path(f'{data_folder}xml_papers/').iterdir() if f.suffixes[0] == ".xml"}

In [ ]:
# Add `is_pdf` and `is_xml` columns to meta data
df['is_pdf'] = df['pmid'].isin(local_pdfs)
df['is_xml'] = df['pmid'].isin(local_xmls)

In [ ]:
# Normalise `cited_by_count` (adding column `cited_count_per_year`)
df['cited_count_per_year'] = df['cited_by_count'] / (current_year - df['pub_year'])

df.head()

## Finding a paper that is nice to highlight in a presentation

Try to find a paper with impact would be nice to use in a demo for 'sematic search'.

In [ ]:
# Filter on peer-reviewed papers of which the pdf is successfully parsed. 
paper_selection = df.loc[(df['is_published'] == True) & (df['is_xml'] == True)]

In [ ]:
# Sort by most 'cited_by_count'
paper_selection.sort_values(['cited_by_count'], ascending=False).head(10)

I think 'cited_by_count' may not be the best parameter for it will have a bias to older papers (which had more chance to be citated). Thus I also added a column 'cited_count_per_year', where the cited count is divided by how many years the paper exists.

In [ ]:
# Sort by most 'cited_count_per_year'
paper_selection.sort_values(['cited_count_per_year'], ascending=False).head(10)

## Proportions by the columns

In [ ]:
df.head()

#### Is Open Acces?

In [ ]:
# # for col in 
# colums = ["is_oa", "is_accepted", "is_published", "is_retracted"]

# col = "is_oa"
# data = df[col].value_counts(normalize=True)
# bottom = np.zeros(3)

# fig, ax = plt.subplots()
# # for idx in data.index:
# p = ax.bar(colums, data.values, 0.7, label=data.index)


In [ ]:
[df[col].value_counts(normalize=True).rename(col) for col in ["is_oa", "is_accepted", "is_published", "is_retracted"]]

In [ ]:
overview = pd.concat(
    [
        df[col].value_counts(normalize=True).rename(col) \
        for col in ["is_oa", "is_accepted", "is_published", "is_retracted"]
    ],
    axis=1
)
overview

In [ ]:
overview.loc[True].values

In [ ]:
fig, ax = plt.subplots()
bottom = np.zeros(4)
for idx in overview.index:
    print(idx, overview.loc[idx].values)
    p = ax.bar(["is_oa", "is_accepted", "is_published", "is_retracted"],
               overview.loc[idx].values, 0.7, 
               label=idx, 
               bottom=bottom
               )
    ax.bar_label(p, label_type='center')

In [ ]:
import numpy as np

In [ ]:
species = ('Adelie', 'Chinstrap', 'Gentoo')
sex_counts = {
    'Male': np.array([73, 34, 61]),
    'Female': np.array([73, 34, 58]),
}
width = 0.6  # the width of the bars: can also be len(x) sequence


fig, ax = plt.subplots()
bottom = np.zeros(3)

for sex, sex_count in sex_counts.items():
    print(sex, sex_count)
    p = ax.bar(species, sex_count, width, label=sex, bottom=bottom)
    bottom += sex_count

    ax.bar_label(p, label_type='center')

ax.set_title('Number of penguins by sex')
ax.legend()

plt.show()

In [ ]:
df['is_oa'].value_counts()

In [ ]:
dict_index = {False: "Closed Access", True: "Open Access"}

In [ ]:
plt.bar(
    [dict_index[idx] for idx in df['is_oa'].value_counts().index] ,
    df['is_oa'].value_counts().values
)


plt.title("Proportions Open Access papers")
plt.ylabel("Count")

plt.show()

In [ ]:
df['is_accepted'].value_counts()

In [ ]:
df['is_retracted'].value_counts()

## Distribution of numerical columns